In [ ]:
import html
import re
import httpx
from google import genai
import os
from dotenv import load_dotenv


load_dotenv()  # Load environment variables from .env file


In [ ]:
from urllib import response


def get_summary(q):
    """Fetches a short summary from Wikipedia based on the search term."""
    headers = {
        "User-Agent": "SummaryExtractorBot/1.0 (your_email@example.com)",
        "Accept-Encoding": "gzip"  # Best practice: Request zipped data to save bandwidth
    }

    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": q,
        "format": "json"
    },
    headers=headers,
    timeout=10.0)

    print("Status:", response.status_code)
    print("Body:", response.text[:500])

    response.raise_for_status()

    # data = response.json()
    # if not data.get("query", {}).get("search"):
    #     raise ValueError(f"No Wikipedia results for: {q}")

    # return data["query"]["search"][0]["snippet"]

    raw_snippet = response.json()["query"]["search"][0]["snippet"]
    clean_snippet = re.sub(r'<[^>]+>', '', raw_snippet)  # Remove HTML tags
    text_snippet = html.unescape(clean_snippet)  # Unescape HTML entities

    return text_snippet



# summary = get_summary("Elon Musk")
# print(summary)


In [ ]:
def quick_math(expr):
    return eval(expr)  # Use eval for quick math evaluation (be cautious with untrusted input)

# eval("2 + 2 * 3")  # Example usage of quick_math





In [ ]:
def fetch_todo(id):
    """Fetches a list of todos from a public API."""
    response = httpx.get(f"https://jsonplaceholder.typicode.com/todos/{id}")
    response.raise_for_status()
    return response.json()

#fetch_todo(2)  # Example usage of fetch_todo

In [ ]:
class Chat:
    def __init__(self, api_key, system=""):
        self.system = system
        self.messages =[]
        self.genai_client= genai.Client(api_key=api_key)
        self.chat = self.genai_client.chats.create(model="gemini-3.5-flash")

        if self.system:
            self.messages.append({"role": "system", "content": self.system})
            self.chat.send_message(self.system)

    def __call__(self, message):
        """Sends a message to the AI model and returns the response."""
        if message:
            self.messages.append({"role": "user", "content": message})

        response = self.chat.send_message(message)
        result = response.text

        return result

In [ ]:
test_chat = Chat(api_key=os.getenv("GEMINI_API_KEY"), system="You are a helpful assistant.")

In [ ]:
#test_chat("what is the capital of france?")
#test_chat("what is 2+5?")
test_chat("Can you fetch me my todo list item with id 1?")  # Example usage of Chat class

In [ ]:
class ReActAgent:
    """
    A ReAct (Reasoning and Acting) agent that can use tools to answer questions.
    
    The agent follows the ReAct pattern:
    1. Reason about the task
    2. Act by calling appropriate tools
    3. Observe the results
    4. Repeat until the task is complete
    """
    
    def __init__(self, api_key):
        self.memory = []
        self.system_prompt = (
            "You are a helpful assistant. You can use the following tools:\n"
            "- get_summary(query: str): to search Wikipedia\n"
            "- quick_math(expr: str): to evaluate a math expression\n"
            "- fetch_todo(): to get a sample todo\n"
            "When you need to use a tool, respond with: Action: <tool_name>[<input>]\n"
            "Otherwise, respond normally as the assistant.\n"
        )
        
        self.chat = Chat(api_key, system=self.system_prompt)

    def __call__(self, message):
        """Process a user message and return a response, using tools if necessary."""
        full_input = "\n".join(self.memory + [f"User: {message}"])
        
        response = self.chat(full_input)
        
        self.memory.append(f"User: {message}")
        self.memory.append(f"Assistant: {response}")

        # Check if the response contains a tool invocation
        if response.startswith("Action:"):
            tool_call = re.match(r'Action:\s*(\w+)\[(.*?)\]', response)
            if not tool_call:
                return "Invalid tool format."

            tool_name, tool_arg = tool_call.groups()
            result = self.invoke_tool(tool_name, tool_arg.strip())
            
            self.memory.append(f"Observation: {result}")
            
            # Recursive call to process the tool result
            return self(f"Observation: {result}")
        else:
            return response

    def invoke_tool(self, name, arg):
        """Execute a tool with the given argument."""
        try:
            if name == "get_summary":
                return get_summary(arg)
            elif name == "quick_math":
                return quick_math(arg)
            elif name == "fetch_todo":
                return fetch_todo([])
            else:
                return f"Unknown tool: {name}"
        except Exception as e:
            return f"Error executing {name}: {str(e)}"

In [ ]:
def main():
    API_KEY = os.getenv("GEMINI_API_KEY")  
    agent = ReActAgent(API_KEY)
    
    print("Welcome to the ReAct Agent! Type 'exit' to quit.")
    print(agent("Hello!"))
    
    print(agent("What is the capital of France?"))
    print(agent("What is 2 + 2?"))
    



if __name__ == "__main__":
    main()